# SPY 1-Minute Data Analysis

This notebook downloads 1 year of SPY 1-minute data from the Financial Modeling Prep (FMP) API, caches it to CSV, explores the data, and creates interactive visualizations using Plotly.

## Cell 1: Imports and Setup

In [1]:
# Core libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from pathlib import Path

# FMP library
from fmp_py.fmp_historical_data import FmpHistoricalData

# Plotting
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Environment variables
from dotenv import load_dotenv
load_dotenv()

True

## Cell 2: Configuration

In [2]:
# Configuration
SYMBOL = 'SPY'
INTERVAL = '1min'
DATA_DIR = Path('../data')  # Relative to notebooks/ directory
CSV_FILE = DATA_DIR / 'spy_1min_1year.csv'

# Date range: 1 year of data
end_date = datetime.now()
start_date = end_date - timedelta(days=365)
FROM_DATE = start_date.strftime('%Y-%m-%d')
TO_DATE = end_date.strftime('%Y-%m-%d')

# Get API key
FMP_API_KEY = os.getenv('FMP_API_KEY')
if not FMP_API_KEY:
    raise ValueError("FMP_API_KEY not found in environment variables")

print(f"Fetching {SYMBOL} {INTERVAL} data from {FROM_DATE} to {TO_DATE}")
print(f"Cache location: {CSV_FILE}")

Fetching SPY 1min data from 2024-12-19 to 2025-12-19
Cache location: ../data/spy_1min_1year.csv


## Cell 3: Download Data from FMP API

In [3]:
# Initialize FMP client
fmp = FmpHistoricalData(api_key=FMP_API_KEY)

# Download data
print("Downloading data from FMP API...")
spy_data = fmp.intraday_history(
    symbol=SYMBOL,
    interval=INTERVAL,
    from_date=FROM_DATE,
    to_date=TO_DATE
)

# Data comes with date as index - keep it that way for time series operations
print(f"Downloaded {len(spy_data):,} rows of data")
print(f"Date range: {spy_data.index.min()} to {spy_data.index.max()}")
print(f"\nColumns: {list(spy_data.columns)}")
print(f"\nFirst row:")
print(spy_data.head(1))

Downloaded 968 rows of data
Date range: 2025-12-17 09:30:00 to 2025-12-19 12:37:00

Columns: ['open', 'low', 'high', 'close', 'volume']

First row:
                       open    low   high   close   volume
date                                                      
2025-12-17 09:30:00  679.88  679.2  679.9  679.54  2188207


## Cell 4: Save to CSV (Caching)

In [4]:
# Ensure data directory exists
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Save to CSV with index (date)
print(f"Saving data to {CSV_FILE}...")
spy_data.to_csv(CSV_FILE)
print(f"✓ Data cached successfully ({CSV_FILE.stat().st_size / 1024 / 1024:.2f} MB)")

Saving data to ../data/spy_1min_1year.csv...
✓ Data cached successfully (0.05 MB)


## Cell 5: Load from CSV (with Cache Check)

In [5]:
# Load data from CSV (or download if not exists)
if CSV_FILE.exists():
    print(f"Loading cached data from {CSV_FILE}...")
    df = pd.read_csv(CSV_FILE)
    
    # Check if date is a column or already the index
    if 'date' in df.columns:
        # Date is a column - set it as index
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date')
    elif not isinstance(df.index, pd.DatetimeIndex):
        # First column is the index but not parsed as datetime
        df.index = pd.to_datetime(df.index)
    
    # Ensure index is named 'date'
    df.index.name = 'date'
    
    print(f"✓ Loaded {len(df):,} rows from cache")
else:
    print("No cached data found. Please run the download cells above.")
    df = None

if df is not None:
    print(f"\nData shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"Index name: {df.index.name}")
    print(f"Index type: {type(df.index).__name__}")
    print(f"Date range: {df.index.min()} to {df.index.max()}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

Loading cached data from ../data/spy_1min_1year.csv...
✓ Loaded 968 rows from cache

Data shape: (968, 5)
Columns: ['open', 'low', 'high', 'close', 'volume']
Index name: date
Index type: DatetimeIndex
Date range: 2025-12-17 09:30:00 to 2025-12-19 12:37:00
Memory usage: 0.04 MB


## Cell 6: Data Exploration - Basic Statistics

In [6]:
if df is not None:
    print("=" * 80)
    print("DATA EXPLORATION - BASIC STATISTICS")
    print("=" * 80)

    # Dataset info
    print("\n1. Dataset Overview:")
    print(f"   Rows: {len(df):,}")
    print(f"   Columns: {len(df.columns)}")
    print(f"   Date Range: {df.index.min()} to {df.index.max()}")
    
    # Calculate unique trading days
    if isinstance(df.index, pd.DatetimeIndex):
        trading_days = df.index.normalize().nunique()
    else:
        trading_days = "N/A (index not datetime)"
    print(f"   Trading Days: {trading_days}")

    # Data types
    print("\n2. Data Types:")
    print(df.dtypes)

    # Missing values
    print("\n3. Missing Values:")
    missing = df.isnull().sum()
    if missing.sum() == 0:
        print("   ✓ No missing values")
    else:
        print(missing[missing > 0])

    # Price statistics
    print("\n4. Price Statistics:")
    print(df[['open', 'high', 'low', 'close', 'volume']].describe())

    # Sample data
    print("\n5. First 5 Rows:")
    display(df.head())

    print("\n6. Last 5 Rows:")
    display(df.tail())

DATA EXPLORATION - BASIC STATISTICS

1. Dataset Overview:
   Rows: 968
   Columns: 5
   Date Range: 2025-12-17 09:30:00 to 2025-12-19 12:37:00
   Trading Days: 3

2. Data Types:
open      float64
low       float64
high      float64
close     float64
volume      int64
dtype: object

3. Missing Values:
   ✓ No missing values

4. Price Statistics:
             open        high         low       close        volume
count  968.000000  968.000000  968.000000  968.000000  9.680000e+02
mean   676.904928  677.076529  676.734669  676.899866  2.458519e+05
std      2.713932    2.697834    2.724183    2.717953  2.991537e+05
min    671.210000  671.440000  671.200000  671.220000  0.000000e+00
25%    674.245000  674.527500  674.045000  674.240000  1.297130e+05
50%    677.595000  677.770000  677.365000  677.565000  1.865140e+05
75%    679.302500  679.492500  679.162500  679.322500  2.667302e+05
max    680.790000  680.830000  680.750000  680.800000  4.965483e+06

5. First 5 Rows:


,open,low,high,close,volume
date,,,,,
2025-12-17 09:30:00,679.88,679.20,679.90,679.54,2188207
2025-12-17 09:31:00,679.59,679.50,680.42,680.08,470791
2025-12-17 09:32:00,680.06,679.81,680.24,679.84,378516
2025-12-17 09:33:00,679.84,679.84,680.31,680.10,316780
2025-12-17 09:34:00,680.10,679.70,680.17,679.78,379027



6. Last 5 Rows:


,open,low,high,close,volume
date,,,,,
2025-12-19 12:33:00,680.48,680.48,680.55,680.51,53991
2025-12-19 12:34:00,680.50,680.44,680.52,680.48,80416
2025-12-19 12:35:00,680.46,680.36,680.46,680.46,253444
2025-12-19 12:36:00,680.46,680.45,680.60,680.52,123974
2025-12-19 12:37:00,680.55,680.50,680.62,680.59,83137


## Cell 7: Plotly Interactive Candlestick Chart

In [7]:
if df is not None:
    # Create figure with secondary y-axis for volume
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=(f'{SYMBOL} 1-Minute Candlestick Chart', 'Volume'),
        row_heights=[0.7, 0.3]
    )

    # Candlestick chart
    fig.add_trace(
        go.Candlestick(
            x=df.index,
            open=df['open'],
            high=df['high'],
            low=df['low'],
            close=df['close'],
            name='OHLC',
            increasing_line_color='green',
            decreasing_line_color='red'
        ),
        row=1, col=1
    )

    # Volume bars
    colors = ['green' if close >= open else 'red'
              for close, open in zip(df['close'], df['open'])]

    fig.add_trace(
        go.Bar(
            x=df.index,
            y=df['volume'],
            name='Volume',
            marker_color=colors,
            opacity=0.5
        ),
        row=2, col=1
    )

    # Update layout
    fig.update_layout(
        title=f'{SYMBOL} 1-Minute Data ({FROM_DATE} to {TO_DATE})',
        yaxis_title='Price ($)',
        yaxis2_title='Volume',
        xaxis_rangeslider_visible=False,
        height=800,
        showlegend=True,
        hovermode='x unified'
    )

    # Update x-axes
    fig.update_xaxes(title_text="Date", row=2, col=1)

    # Show the plot
    fig.show()

    print("\n✓ Interactive chart created successfully")
    print("  - Use mouse to zoom, pan, and hover for details")
    print("  - Double-click to reset view")


✓ Interactive chart created successfully
  - Use mouse to zoom, pan, and hover for details
  - Double-click to reset view
